# Notebook 02: OpenRouter Price Check

## Purpose

This notebook executes, **on Kaggle only**, a metadata-level price
verification of OpenRouter models to update the benchmark model pool and
resolve Q-002.

It checks current public prices for candidate small/cheap language models,
applies strict eligibility thresholds, and writes a verified
`configs/model_pool.yaml`.

It does **not** call OpenRouter inference, does **not** spend tokens, does
**not** use a paid API key, and does **not** use GPU. It only reads the public
model catalog endpoint.


## Kaggle-only policy

- Execute **only on Kaggle**. Do not run locally.
- CPU only. No GPU.
- Internet enabled (public catalog only).
- No OpenRouter inference. No chat completions. No paid calls.
- No token spend. No API key required (public catalog).
- No model downloads. No training.
- Only public pricing metadata is consulted.


## 1. Environment setup


In [ ]:
import sys
import platform
import subprocess
import datetime

print("Python:", sys.version)
print("Platform:", platform.platform())

try:
    import requests
    print("requests:", requests.__version__)
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests"])
    import requests
    print("requests installed:", requests.__version__)

try:
    import yaml
    print("pyyaml available")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml"])
    import yaml
    print("pyyaml installed")

import json
import os
import time
import math
from pathlib import Path

EXECUTED_AT = datetime.datetime.now(datetime.timezone.utc).isoformat()
print("Executed at (UTC):", EXECUTED_AT)


## 2. Project path detection


In [ ]:
CANDIDATES = [
    Path("/kaggle/input/slm-efficiency-frontier"),
    Path("/kaggle/working/slm-efficiency-frontier"),
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]

PROJECT_ROOT = None
for candidate in CANDIDATES:
    if (candidate / "work" / "1_research").exists() or (candidate / ".core").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()

RESEARCH_ARTIFACTS_DIR = PROJECT_ROOT / "work" / "1_research"
REVIEW_DIR = PROJECT_ROOT / "work" / "4_for-review"
GITHUB_DIR = PROJECT_ROOT / "work" / "5_final-work" / "github"
RELEASE_DIR = PROJECT_ROOT / "artifacts/release"
CONFIGS_DIR = GITHUB_DIR / "configs"

for d in (RESEARCH_ARTIFACTS_DIR, REVIEW_DIR, GITHUB_DIR, RELEASE_DIR, CONFIGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

MODEL_POOL_PATH = CONFIGS_DIR / "model_pool.yaml"
RESULTS_PATH = RESEARCH_ARTIFACTS_DIR / "openrouter_model_sweep_results.json"
REPORT_PATH = RESEARCH_ARTIFACTS_DIR / "openrouter_model_sweep.md"
LEGITIMACY_PATH = RESEARCH_ARTIFACTS_DIR / "legitimacy_qa_report.md"
SNAPSHOT_PATH = REVIEW_DIR / "research_snapshot.md"
QA_FINDINGS_PATH = RELEASE_DIR / "qa_findings.md"
RESEARCH_FINDINGS_PATH = RELEASE_DIR / "research_findings.md"
CHANGE_LOG_PATH = RELEASE_DIR / "change_log.md"
OPEN_QUESTIONS_PATH = RELEASE_DIR / "open_questions.md"
DECISIONS_PATH = RELEASE_DIR / "decisions.md"

print("PROJECT_ROOT:", PROJECT_ROOT)


## 3. OpenRouter metadata source

The public model catalog endpoint returns model metadata including pricing.
No authentication is required. Pricing fields are USD per token and are
converted to USD per 1 million tokens.


In [ ]:
OPENROUTER_MODELS_URL = "https://openrouter.ai/api/v1/models"
PRICE_SOURCE_URL = OPENROUTER_MODELS_URL

# Strict eligibility thresholds (exclusive).
MAX_INPUT_PRICE_PER_MILLION = 0.12
MAX_OUTPUT_PRICE_PER_MILLION = 0.30

# Seed candidates from the current model pool.
SEED_CANDIDATES = [
    "deepseek/deepseek-v4-flash",
    "qwen/qwen-2.5-7b-instruct",
    "meta-llama/llama-3.2-3b-instruct",
    "google/gemma-2-9b-it",
    "mistralai/mistral-7b-instruct",
]

# Notebook 01 outcome (real, executed on Kaggle).
NB01_MAX_SCORE = 45
NB01_ADJACENT = 13
NB01_DECISION = "PROCEED"


## 4. Load current model pool


In [ ]:
def load_current_model_pool():
    if not MODEL_POOL_PATH.exists():
        print("No existing model_pool.yaml found; using seed candidates only.")
        return {}
    try:
        data = yaml.safe_load(MODEL_POOL_PATH.read_text(encoding="utf-8")) or {}
        models = {m["model_id"]: m for m in data.get("models", []) if "model_id" in m}
        print(f"Loaded {len(models)} models from existing model_pool.yaml")
        return models
    except Exception as exc:
        print("Failed to load model_pool.yaml:", exc)
        return {}


current_pool = load_current_model_pool()


## 5. Fetch OpenRouter model catalog


In [ ]:
def fetch_catalog(url=OPENROUTER_MODELS_URL, timeout=60):
    print("Fetching", url)
    resp = requests.get(url, timeout=timeout, headers={"Accept": "application/json"})
    resp.raise_for_status()
    payload = resp.json()
    data = payload.get("data", [])
    print(f"Catalog entries: {len(data)}")
    return data


catalog = fetch_catalog()
catalog_by_id = {m.get("id"): m for m in catalog if m.get("id")}
print("Unique catalog ids:", len(catalog_by_id))


## 6. Normalize pricing


In [ ]:
def to_float(value):
    if value is None:
        return None
    try:
        f = float(value)
    except (TypeError, ValueError):
        return None
    if math.isnan(f) or math.isinf(f):
        return None
    return f


def normalize_model(model_obj):
    """Return a normalized record with USD-per-million prices or None."""
    if model_obj is None:
        return None
    pricing = model_obj.get("pricing") or {}
    # OpenRouter pricing is USD per token; convert to USD per 1M tokens.
    prompt_per_token = to_float(pricing.get("prompt"))
    completion_per_token = to_float(pricing.get("completion"))
    input_pm = prompt_per_token * 1_000_000 if prompt_per_token is not None else None
    output_pm = completion_per_token * 1_000_000 if completion_per_token is not None else None
    return {
        "model_id": model_obj.get("id"),
        "name": model_obj.get("name"),
        "provider": (model_obj.get("id") or "").split("/")[0] if model_obj.get("id") else None,
        "context_window": model_obj.get("context_length"),
        "input_price_per_million": input_pm,
        "output_price_per_million": output_pm,
        "price_source_url": PRICE_SOURCE_URL,
        "price_checked_at": EXECUTED_AT,
    }


## 7. Candidate discovery


In [ ]:
# Substrings that suggest small/cheap models. Checked against lowercased id
# and name. Large models that match by accident are filtered by price later.
SMALL_KEYWORDS = [
    "smollm", "tinyllama", "phi-3", "phi3", "phi-3.5", "mini", "small",
    "gemma-2-2b", "gemma-2-9b", "gemma-3-1b", "gemma-3-4b", "gemma-3-12b",
    "llama-3.2-1b", "llama-3.2-3b", "llama-3.1-8b",
    "qwen2.5-0.5b", "qwen2.5-1.5b", "qwen2.5-3b", "qwen2.5-7b", "qwen2.5-14b",
    "qwen3-0.6b", "qwen3-1.7b", "qwen3-4b", "qwen3-8b",
    "ministral-3b", "ministral-7b", "mistral-7b",
    "1b", "1.5b", "1.7b", "2b", "3b", "4b", "7b", "8b", "9b", "0.5b",
]


def is_small_candidate(model_id, name):
    text = f"{model_id} {name}".lower()
    return any(kw in text for kw in SMALL_KEYWORDS)


def discover_candidates(catalog):
    discovered = []
    for model_obj in catalog:
        mid = model_obj.get("id")
        if not mid:
            continue
        if is_small_candidate(mid, model_obj.get("name")):
            discovered.append(mid)
    return sorted(set(discovered))


discovered_ids = discover_candidates(catalog)
print("Discovered small/cheap candidate ids:", len(discovered_ids))


## 8. Eligibility rules


In [ ]:
def eligibility(norm):
    """Apply strict thresholds. Returns (eligible, reason)."""
    if norm is None:
        return False, "not found in catalog"
    inp = norm["input_price_per_million"]
    out = norm["output_price_per_million"]
    if inp is None or out is None:
        return False, "price missing or ambiguous"
    if inp >= MAX_INPUT_PRICE_PER_MILLION:
        return False, f"input price {inp} >= {MAX_INPUT_PRICE_PER_MILLION}"
    if out >= MAX_OUTPUT_PRICE_PER_MILLION:
        return False, f"output price {out} >= {MAX_OUTPUT_PRICE_PER_MILLION}"
    return True, "verified below thresholds"


def build_entry(model_id):
    norm = normalize_model(catalog_by_id.get(model_id))
    if norm is None:
        return {
            "model_id": model_id,
            "provider": model_id.split("/")[0] if "/" in model_id else None,
            "context_window": None,
            "input_price_per_million": None,
            "output_price_per_million": None,
            "price_source_url": None,
            "price_checked_at": EXECUTED_AT,
            "eligible": False,
            "eligibility_reason": "not found in catalog",
            "notes": "seed candidate not present in OpenRouter catalog",
        }
    eligible, reason = eligibility(norm)
    return {
        "model_id": norm["model_id"],
        "provider": norm["provider"],
        "context_window": norm["context_window"],
        "input_price_per_million": norm["input_price_per_million"],
        "output_price_per_million": norm["output_price_per_million"],
        "price_source_url": norm["price_source_url"],
        "price_checked_at": norm["price_checked_at"],
        "eligible": eligible,
        "eligibility_reason": reason,
        "notes": "",
    }


# Build the final candidate set: seed candidates first, then discovered
# candidates that are not already seeds.
candidate_ids = list(SEED_CANDIDATES)
for mid in discovered_ids:
    if mid not in candidate_ids:
        candidate_ids.append(mid)

# Cap discovered candidates to keep the pool focused; keep all seeds.
MAX_DISCOVERED_KEPT = 60
seed_set = set(SEED_CANDIDATES)
kept = list(SEED_CANDIDATES)
discovered_kept = [m for m in discovered_ids if m not in seed_set][:MAX_DISCOVERED_KEPT]
kept.extend(discovered_kept)

entries = [build_entry(mid) for mid in kept]
print("Total pool entries:", len(entries))


## 9. Update model_pool.yaml


In [ ]:
def write_model_pool(entries):
    # Order: eligible first, then ineligible by price, then unknown.
    def sort_key(e):
        eligible = 0 if e["eligible"] else 1
        inp = e["input_price_per_million"]
        inp_sort = inp if inp is not None else 9e9
        return (eligible, inp_sort, e["model_id"])

    ordered = sorted(entries, key=sort_key)
    lines = []
    lines.append("# OpenRouter model pool.")
    lines.append("# Updated by Kaggle notebook 02_kaggle_openrouter_price_check.ipynb.")
    lines.append("# Eligibility requires input < 0.12 USD/1M and output < 0.30 USD/1M (exclusive).")
    lines.append("# Unknown/missing/ambiguous prices => eligible: false.")
    lines.append("")
    lines.append("models:")
    for e in ordered:
        lines.append(f'  - model_id: "{e["model_id"]}"')
        lines.append(f'    provider: "{e["provider"] or ""}"')
        lines.append(f'    context_window: {e["context_window"] if e["context_window"] is not None else "null"}')
        inp = e["input_price_per_million"]
        out = e["output_price_per_million"]
        lines.append(f'    input_price_per_million: {inp if inp is not None else "null"}')
        lines.append(f'    output_price_per_million: {out if out is not None else "null"}')
        lines.append(f'    price_source_url: "{e["price_source_url"] or ""}"')
        lines.append(f'    price_checked_at: "{e["price_checked_at"] or ""}"')
        lines.append(f'    eligible: {"true" if e["eligible"] else "false"}')
        lines.append(f'    eligibility_reason: "{e["eligibility_reason"]}"')
        lines.append(f'    notes: "{e["notes"]}"')
    MODEL_POOL_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("model_pool.yaml written to", MODEL_POOL_PATH)


write_model_pool(entries)


## 10. Save structured results


In [ ]:
eligible_entries = [e for e in entries if e["eligible"]]
ineligible_entries = [e for e in entries if not e["eligible"]]
unknown_entries = [e for e in entries if e["input_price_per_million"] is None or e["output_price_per_million"] is None]


def deepseek_status():
    for e in entries:
        if e["model_id"] == "deepseek/deepseek-v4-flash":
            return e
    return None


ds = deepseek_status()

results = {
    "executed_at": EXECUTED_AT,
    "kaggle_only": True,
    "openrouter_inference_used": False,
    "gpu_used": False,
    "source_endpoint": OPENROUTER_MODELS_URL,
    "thresholds": {
        "max_input_price_per_million_usd": MAX_INPUT_PRICE_PER_MILLION,
        "max_output_price_per_million_usd": MAX_OUTPUT_PRICE_PER_MILLION,
    },
    "catalog_entries": len(catalog),
    "total_pool_entries": len(entries),
    "eligible_count": len(eligible_entries),
    "ineligible_count": len(ineligible_entries),
    "unknown_or_ambiguous_count": len(unknown_entries),
    "deepseek_v4_flash": ds,
    "eligible_models": eligible_entries,
    "ineligible_models": ineligible_entries,
}

with open(RESULTS_PATH, "w", encoding="utf-8") as fh:
    json.dump(results, fh, indent=2, allow_nan=False)
print("Results written to", RESULTS_PATH)
print("Eligible:", len(eligible_entries), "Ineligible:", len(ineligible_entries))


## 11. Generate markdown report


In [ ]:
def model_pool_validity(n_eligible):
    if n_eligible == 0:
        return 5
    if n_eligible <= 2:
        return 7
    if n_eligible <= 4:
        return 8
    return 9


def build_report():
    n_eligible = len(eligible_entries)
    mpv = model_pool_validity(n_eligible)
    L = []
    L.append("# OpenRouter Model Sweep Report")
    L.append("")
    L.append(f"**Execution timestamp (UTC):** {EXECUTED_AT}")
    L.append("")
    L.append("## Methodology")
    L.append("")
    L.append(
        "Metadata-level price verification using the public OpenRouter model "
        f"catalog endpoint `{OPENROUTER_MODELS_URL}`. No inference, no paid "
        "calls, no GPU. Prices are USD per token and converted to USD per 1 "
        "million tokens. Strict eligibility: input < 0.12 and output < 0.30 "
        "USD/1M (exclusive). Missing/ambiguous/non-numeric prices are "
        "ineligible."
    )
    L.append("")
    L.append("## Source endpoint")
    L.append("")
    L.append(f"`{OPENROUTER_MODELS_URL}`")
    L.append("")
    L.append("## Eligibility thresholds")
    L.append("")
    L.append(f"- input_price_per_million_usd < {MAX_INPUT_PRICE_PER_MILLION}")
    L.append(f"- output_price_per_million_usd < {MAX_OUTPUT_PRICE_PER_MILLION}")
    L.append("")
    L.append("## Current model pool results")
    L.append("")
    L.append("| model_id | input $/1M | output $/1M | eligible | reason |")
    L.append("|----------|-----------|------------|----------|--------|")
    for e in entries:
        inp = e["input_price_per_million"]
        out = e["output_price_per_million"]
        L.append(f'| {e["model_id"]} | {inp} | {out} | {e["eligible"]} | {e["eligibility_reason"]} |')
    L.append("")
    L.append("## Discovered candidate models")
    L.append("")
    L.append(f"- Discovered small/cheap candidate ids: {len(discovered_ids)}")
    L.append(f"- Kept in pool (seeds + discovered cap {MAX_DISCOVERED_KEPT}): {len(entries)}")
    L.append("")
    L.append("## Eligible models")
    L.append("")
    if eligible_entries:
        for e in eligible_entries:
            L.append(f'- {e["model_id"]} | input {e["input_price_per_million"]} | output {e["output_price_per_million"]}')
    else:
        L.append("None.")
    L.append("")
    L.append("## Ineligible models")
    L.append("")
    if ineligible_entries:
        for e in ineligible_entries:
            L.append(f'- {e["model_id"]} | {e["eligibility_reason"]} | input {e["input_price_per_million"]} | output {e["output_price_per_million"]}')
    else:
        L.append("None.")
    L.append("")
    L.append("## Unknown/ambiguous price models")
    L.append("")
    if unknown_entries:
        for e in unknown_entries:
            L.append(f'- {e["model_id"]} | {e["eligibility_reason"]}')
    else:
        L.append("None.")
    L.append("")
    L.append("## DeepSeek V4 Flash baseline status")
    L.append("")
    if ds:
        L.append(f'- model_id: {ds["model_id"]}')
        L.append(f'- input $/1M: {ds["input_price_per_million"]}')
        L.append(f'- output $/1M: {ds["output_price_per_million"]}')
        L.append(f'- eligible: {ds["eligible"]}')
        L.append(f'- reason: {ds["eligibility_reason"]}')
    else:
        L.append("- Not found in catalog.")
    L.append("")
    L.append("## Enough eligible models for a meaningful benchmark")
    L.append("")
    if n_eligible >= 3:
        L.append(f"Yes — {n_eligible} eligible models are available.")
    elif n_eligible >= 1:
        L.append(f"Marginal — only {n_eligible} eligible model(s). More candidates recommended.")
    else:
        L.append("No — zero eligible models. The benchmark cannot run a real evaluation yet.")
    L.append("")
    L.append("## Recommended decision")
    L.append("")
    if n_eligible >= 3:
        decision = "PROCEED"
    elif n_eligible >= 1:
        decision = "REVISE"
    else:
        decision = "STOP"
    L.append(f"**{decision}**")
    L.append("")
    L.append("## Required changes before notebook 03")
    L.append("")
    if decision == "STOP":
        L.append("- Add more candidate small/cheap models and re-run price verification.")
    elif decision == "REVISE":
        L.append("- Expand the candidate pool to reach at least 3 eligible models.")
        L.append("- Keep all unverified models as eligible: false.")
    else:
        L.append("- None required. Proceed to dry-run evaluation (notebook 03).")
    L.append("")
    return "\n".join(L), decision, mpv, n_eligible


report_text, decision, mpv, n_eligible = build_report()
with open(REPORT_PATH, "w", encoding="utf-8") as fh:
    fh.write(report_text)
print("Report written to", REPORT_PATH)
print("Decision:", decision, "| eligible:", n_eligible, "| model_pool_validity:", mpv)


## 12. Update legitimacy QA


In [ ]:
def update_legitimacy(decision, mpv, n_eligible):
    total = 8 + 9 + 9 + 9 + 8 + 8 + 9 + 9 + mpv + 7
    content = []
    content.append("# Legitimacy QA Report")
    content.append("")
    content.append("## Status: UPDATED BY NOTEBOOKS 01 + 02 (Kaggle execution)")
    content.append("")
    content.append(f"**Last update (UTC):** {EXECUTED_AT}")
    content.append("")
    content.append("## Duplication sweep outcome (notebook 01)")
    content.append("")
    content.append(f"- Max duplication risk score: {NB01_MAX_SCORE}")
    content.append(f"- Adjacent projects: {NB01_ADJACENT}")
    content.append(f"- Near/exact duplicates: 0")
    content.append(f"- Duplication decision: {NB01_DECISION}")
    content.append("")
    content.append("## OpenRouter price verification outcome (notebook 02)")
    content.append("")
    content.append(f"- Eligible models: {n_eligible}")
    content.append(f"- Model pool validity component: {mpv}/10")
    content.append(f"- Price-check decision: {decision}")
    content.append("")
    content.append("## Dimension scores")
    content.append("")
    content.append("| Dimension | Score (/10) | Rationale |")
    content.append("|-----------|-------------|-----------|")
    content.append(f"| originality | 8 | adjacent projects only; no duplicate |")
    content.append("| practical usefulness | 9 | Directly useful for model selection. |")
    content.append("| benchmark clarity | 9 | Clear metric, tasks, validators. |")
    content.append("| automatic evaluability | 9 | All v0.1 tasks auto-scorable. |")
    content.append("| PyTorch centrality | 8 | Evaluator, baselines, metrics use PyTorch. |")
    content.append("| Hugging Face fit | 8 | Dataset, Space, model repos planned. |")
    content.append("| Kaggle reproducibility | 9 | Kaggle-first notebooks and configs. |")
    content.append("| cost discipline | 9 | Hard caps, guards, dry-run default. |")
    content.append(f"| model pool validity | {mpv} | {n_eligible} eligible models verified on Kaggle. |")
    content.append("| release readiness | 7 | Skeletons ready; pending real-call implementation. |")
    content.append("")
    content.append(f"**Total:** {total}/100")
    content.append("")
    if total >= 85:
        content.append("Verdict: ACCEPTED (>=85). The project passes the legitimacy gate.")
    else:
        content.append(f"Verdict: BELOW 85 ({total}/100). Not yet accepted. Pending more eligible models or real-call implementation.")
    content.append("")
    with open(LEGITIMACY_PATH, "w", encoding="utf-8") as fh:
        fh.write("\n".join(content))
    print("Legitimacy report updated. Total:", total)
    return total


total = update_legitimacy(decision, mpv, n_eligible)


## 13. Update memory and QA


In [ ]:
def update_memory_and_qa(decision, total, n_eligible):
    # research_findings.md
    rf = []
    rf.append("# Research Findings")
    rf.append("")
    rf.append("| ID | Category | Source | Summary | Status |")
    rf.append("|----|----------|--------|---------|--------|")
    rf.append("| F-001 | theme | internal | SLM Efficiency Frontier selected as benchmark theme. | confirmed |")
    rf.append(f"| F-002 | hf_duplication | Kaggle notebook 01 | HF duplication sweep executed; max risk {NB01_MAX_SCORE}; {NB01_ADJACENT} adjacent; 0 duplicates. | confirmed |")
    rf.append(f"| F-003 | openrouter_models | Kaggle notebook 02 | Price verification executed at {EXECUTED_AT}; {n_eligible} eligible models. | confirmed |")
    rf.append("| F-004 | literature | pending | Literature sweep not yet executed. | pending |")
    with open(RESEARCH_FINDINGS_PATH, "w", encoding="utf-8") as fh:
        fh.write("\n".join(rf) + "\n")

    # qa_findings.md
    qa = []
    qa.append("# QA Findings")
    qa.append("")
    qa.append("| ID | Area | Severity | Description | Status |")
    qa.append("|----|------|----------|-------------|--------|")
    qa.append(f"| Q-001 | research | BLOCKER | HF duplication sweep executed on Kaggle; max risk {NB01_MAX_SCORE}; 0 duplicates. | resolved |")
    qa.append(f"| Q-002 | budget | MAJOR | OpenRouter price verification executed on Kaggle at {EXECUTED_AT}; {n_eligible} eligible models. | resolved |")
    qa.append("| Q-003 | metrics | MINOR | Determinism tests not yet run (Kaggle-only). | open |")
    qa.append("| Q-015 | runtime | MAJOR | Real OpenRouter execution (_real_call) not implemented. | open |")
    with open(QA_FINDINGS_PATH, "w", encoding="utf-8") as fh:
        fh.write("\n".join(qa) + "\n")

    # open_questions.md
    oq = []
    oq.append("# Open Questions")
    oq.append("")
    oq.append(f"- OQ-001: HF duplication sweep executed on Kaggle. Max risk {NB01_MAX_SCORE}; PROCEED. Resolved.")
    oq.append(f"- OQ-002: OpenRouter price verification executed at {EXECUTED_AT}; {n_eligible} eligible models. Resolved.")
    oq.append("- OQ-003: Final public/private split ratio pending dataset curator design.")
    oq.append("- OQ-004: PyTorch baseline architectures for calibration pending baseline engineer.")
    oq.append("- OQ-005: Space demo cached leaderboard vs live dry-run pending design confirmation.")
    oq.append("- OQ-006: Real OpenRouter execution (_real_call) not implemented; pending notebook 04.")
    with open(OPEN_QUESTIONS_PATH, "w", encoding="utf-8") as fh:
        fh.write("\n".join(oq) + "\n")

    # change_log.md (append)
    entry = (
        f"\n## {EXECUTED_AT} (Kaggle notebook 02 executed)\n"
        "- Executed OpenRouter price verification on Kaggle (CPU, internet, no "
        "inference, no GPU, no paid calls).\n"
        f"- Catalog entries: {len(catalog)}; pool entries: {len(entries)}; "
        f"eligible: {n_eligible}; ineligible: {len(ineligible_entries)}.\n"
        f"- Decision: {decision}. Legitimacy QA total: {total}/100.\n"
        "- Updated model_pool.yaml, openrouter_model_sweep_results.json, "
        "openrouter_model_sweep.md, legitimacy_qa_report.md, research_snapshot.md, "
        "research_findings.md, qa_findings.md, open_questions.md, decisions.md.\n"
        "- Marked Q-002 resolved after real Kaggle execution.\n"
    )
    with open(CHANGE_LOG_PATH, "a", encoding="utf-8") as fh:
        fh.write(entry)

    # decisions.md (append)
    dr = (
        f"\n## DR-013: OpenRouter price verification result ({EXECUTED_AT}, Kaggle)\n"
        f"- **Decision:** {decision}.\n"
        f"- **Rationale:** Kaggle notebook 02 verified public OpenRouter prices. "
        f"{n_eligible} eligible models under strict thresholds (input < 0.12, "
        f"output < 0.30 USD/1M). All unknown/ambiguous prices remain ineligible.\n"
        f"- **Consequences:** Q-002 resolved. Legitimacy QA total {total}/100.\n"
        "- **Next action:** Proceed to notebook 03 (dry-run) if enough eligible models; "
        "otherwise expand candidate pool.\n"
    )
    with open(DECISIONS_PATH, "a", encoding="utf-8") as fh:
        fh.write(dr)

    print("Memory and QA files updated.")


update_memory_and_qa(decision, total, n_eligible)


## 14. Review research snapshot update


In [ ]:
def update_snapshot(decision, total, n_eligible):
    snap = []
    snap.append("# Research Snapshot")
    snap.append("")
    snap.append(f"**Updated by Kaggle notebook 02 at (UTC):** {EXECUTED_AT}")
    snap.append("")
    snap.append("## Theme")
    snap.append("SLM Efficiency Frontier (selected). v0.1 is strictly English-only.")
    snap.append("")
    snap.append("## HF duplication sweep (notebook 01)")
    snap.append(f"- Max duplication risk score: {NB01_MAX_SCORE}; {NB01_ADJACENT} adjacent; 0 duplicates.")
    snap.append(f"- Decision: {NB01_DECISION}.")
    snap.append("")
    snap.append("## OpenRouter price verification (notebook 02)")
    snap.append(f"- Executed: yes (Kaggle, {EXECUTED_AT}).")
    snap.append(f"- Eligible models: {n_eligible}.")
    snap.append(f"- Decision: {decision}.")
    snap.append(f"- Legitimacy QA total: {total}/100.")
    snap.append("")
    snap.append("## Pending")
    snap.append("- Real OpenRouter execution implementation (notebook 04).")
    snap.append("- Final runtime QA (notebook 05).")
    with open(SNAPSHOT_PATH, "w", encoding="utf-8") as fh:
        fh.write("\n".join(snap) + "\n")
    print("Research snapshot updated:", SNAPSHOT_PATH)


update_snapshot(decision, total, n_eligible)


## Next steps

1. Inspect `artifacts/research/openrouter_model_sweep.md` and the eligible model
   list.
2. If fewer than 3 eligible models, expand candidates and re-run notebook 02.
3. If enough eligible models, proceed to notebook 03 (dry-run evaluation).
4. Never run real inference until `_real_call()` is implemented in notebook 04.

This notebook does not call OpenRouter inference, does not spend tokens, does
not use GPU, and does not download models. It is public-catalog metadata only.
